# Cleaning Bay — Fix Your Data

Fix the problems found in the Inspection bay. Delete the blocks you don't need and keep the ones that apply to your dataset.

See the [Cleaning README](README.md) for full documentation.

In [ ]:
# ============================================================
# SALON CONFIG — edit the line below to point to your CSV file
# ============================================================
import pandas as pd
from pathlib import Path

PARKING_BAY = Path("../../parking_bay")
CSV_FILE = "your_file.csv"  # <-- change this to your file name

df = pd.read_csv(PARKING_BAY / CSV_FILE)
print(f"Loaded: {CSV_FILE}  |  Shape: {df.shape}")

## Drop Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate row(s). Rows remaining: {after}")

## Drop Rows with Missing Values

Drops any row that has at least one null. Use `subset=` to only check specific columns.

In [ ]:
before = len(df)

# Drop rows with ANY null:
df = df.dropna()

# Or drop rows with nulls only in specific columns:
# df = df.dropna(subset=["column_name", "another_column"])

after = len(df)
print(f"Removed {before - after} row(s) with missing values. Rows remaining: {after}")

## Fill Missing Values

Instead of dropping, fill nulls with a value. Uncomment the strategy that fits your use case.

In [ ]:
# Fill all nulls with a fixed value:
# df = df.fillna(0)
# df = df.fillna("unknown")

# Fill nulls in a specific column:
# df["column_name"] = df["column_name"].fillna(0)
# df["column_name"] = df["column_name"].fillna(df["column_name"].median())
# df["column_name"] = df["column_name"].fillna(df["column_name"].mode()[0])

# Forward fill (carry last known value forward):
# df = df.ffill()

# Backward fill:
# df = df.bfill()

print("Remaining nulls per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## Standardize Column Names

Strip whitespace and convert to snake_case so column names are consistent and easy to work with.

In [ ]:
original_cols = df.columns.tolist()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

print("Column name changes:")
for old, new in zip(original_cols, df.columns):
    if old != new:
        print(f"  '{old}'  →  '{new}'")
    else:
        print(f"  '{old}'  (unchanged)")

## Fix Data Types

Convert columns to their correct types. Update the column names and target types below.

In [ ]:
# Convert to numeric (errors='coerce' turns unparseable values into NaN)
# df["some_column"] = pd.to_numeric(df["some_column"], errors="coerce")

# Convert to datetime
# df["date_column"] = pd.to_datetime(df["date_column"], errors="coerce")

# Convert to integer (only safe if no nulls in that column)
# df["id_column"] = df["id_column"].astype(int)

# Convert to string
# df["code_column"] = df["code_column"].astype(str)

print("Current dtypes:")
print(df.dtypes)

## Clean String Columns

Strip leading/trailing whitespace and normalize casing.

In [ ]:
string_cols = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]

for col in string_cols:
    df[col] = df[col].str.strip()

print(f"Stripped whitespace from {len(string_cols)} string column(s): {string_cols}")

# To also lowercase all string columns:
# for col in string_cols:
#     df[col] = df[col].str.lower()

## Filter Rows

Remove rows that don't meet your criteria. Uncomment and adjust the condition.

In [ ]:
before = len(df)

# Keep rows where condition is True:
# df = df[df["column_name"] > 0]
# df = df[df["status"] == "active"]
# df = df[df["date"] >= "2024-01-01"]
# df = df[df["column_name"].notna()]

# Exclude specific values:
# df = df[~df["column_name"].isin(["bad_value", "another_bad_value"])]

after = len(df)
print(f"Filtered out {before - after} row(s). Rows remaining: {after}")

## Drop Columns

Remove columns you don't need.

In [ ]:
# Drop specific columns:
# df = df.drop(columns=["column_to_drop", "another_column"])

# Drop columns that are entirely null:
# df = df.dropna(axis=1, how='all')

print(f"Remaining columns ({len(df.columns)}): {df.columns.tolist()}")

## Final Check

Quick look at the cleaned DataFrame before moving on.

In [ ]:
print(f"Shape: {df.shape}")
print(f"Nulls remaining: {df.isnull().sum().sum()}")
print(f"Duplicates remaining: {df.duplicated().sum()}")
df.head()

---
## Script Generator

Delete any blocks above that you don't want in the output, then run this cell.
A clean `.py` script will be saved to the `scripts/` folder.

In [ ]:
# ============================================================
# SCRIPT GENERATOR
# ============================================================
import nbformat
from pathlib import Path

NOTEBOOK_NAME = "cleaning.ipynb"

nb = nbformat.read(NOTEBOOK_NAME, as_version=4)

code_cells = [
    cell for cell in nb.cells
    if cell.cell_type == "code"
    and cell.source.strip()
    and "# SCRIPT GENERATOR" not in cell.source
]

script = "\n\n".join(cell.source for cell in code_cells)

output_dir = Path("../../scripts")
output_dir.mkdir(exist_ok=True)
output_file = output_dir / f"{Path(NOTEBOOK_NAME).stem}_script.py"
output_file.write_text(script, encoding="utf-8")

print(f"Script saved to: {output_file.resolve()}")